## WEEK 6 - Embeddings & Vector Databases

In [106]:
from transformers import logging
logging.set_verbosity_error()

In [ ]:
#  FAISS

In [9]:
from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer('all-MiniLM-L6-v2')

docs = ["I love AI", "Machine learning is powerful"]

# Convert to vectors
vectors = model.encode(docs)

# Create index
index = faiss.IndexFlatL2(384)
index.add(vectors)

# Query
query = model.encode([" ML is subset of AI"])
D, I = index.search(query, k=1)

print(docs[I[0][0]])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1869.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I love AI


In [ ]:
from huggingface_hub import list_models

models = list_models()

for model in models:
    if "sentence-transformers" in model.modelId:
        print(model.modelId)

In [ ]:
# SBERT(sentence BERT) - Generate sentence embedding

In [14]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

emb1 = model.encode("Blue is color")
emb2 = model.encode("color is blue")

similarity = util.cos_sim(emb1, emb2)
similarity

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4573.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[0.9650]])

In [13]:
# chroma vector DB

In [ ]:
% pip install langchain chromadb sentence-transformers langchain-community langchain-huggingface

In [ ]:
# Exercise:2 Semantic search engine using chromaDB

In [13]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

docs = ["AI is most important field in future", "Python is popular","ML is subset of AI","Hdfc bank is very expensive","sabarmati river bank is budget friendly"]

db = Chroma.from_texts(
    docs,
    embedding,
    collection_name="my_collection"
)

results = db.similarity_search("which is most scalable field? ", k=3)

for r in results:
    print(r.page_content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4238.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AI is most important field in future
Python is popular
Hdfc bank is very expensive


In [ ]:
%pip install pypdf

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings as HE


/home/shrutik/Srutik/12-Week-Internship/.rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [113]:
loader = PyPDFLoader("/home/shrutik/Downloads/aiml.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 10
)

chunks = text_splitter.split_documents(documents)
print(len(chunks))


20


In [107]:
embedding = HE(model="sentence-transformers/all-MiniLM-L6-V2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5281.27it/s]


In [110]:

db = Chroma.from_documents(chunks,
                           embedding,
                           persist_directory=None,
                           collection_name="new_collection_1" )


In [111]:
query = "What is embeddings?"

results = db.similarity_search(query,k=3)

for i in results:
    print("----")
    print(i.page_content)
print(len(documents))

----
Chunking: Chunking is the process of splitting large text into smaller parts before embedding.
----
Embeddings: Embeddings convert text into numerical vectors that capture semantic meaning.
----
Embeddings: Embeddings convert text into numerical vectors that capture semantic meaning.
Similar texts produce similar vectors. Used in semantic search, clustering, and recommendations.
Vector Databases: Vector databases store embeddings and enable fast similarity search. Tools like
1


In [90]:
loader = PyPDFLoader("/home/shrutik/Downloads/health.pdf")
doc = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 150,chunk_overlap=50)

chunks = text_splitter.split_documents(doc)
print(len(chunks))

21


In [108]:
db = Chroma.from_documents(chunks,
                           embedding,
                           collection_name="health")

In [ ]:
query = "what is immunity?"

results = db.similarity_search(query,k= 3)

for i in results:
    print("-"*50)
    print(i.page_content)
    print(i.metadata)

--------------------------------------------------
key aspects.
Hygiene: Hygiene involves practices that maintain cleanliness and prevent disease. Regular 
handwashing, dental care, and personal cleanliness reduce the risk of infections.
Immunity: Immunity is the body’s ability to defend against infections and diseases. A strong
{'page': 0, 'producer': 'LibreOffice 7.3', 'source': '/home/shrutik/Downloads/health.pdf', 'creationdate': '2026-03-26T12:09:44+05:30', 'total_pages': 1, 'Topic': 'Health', 'creator': 'Writer', 'File': '/home/shrutik/Downloads/health.pdf', 'Dept': 'Wellness', 'page_label': '1'}
--------------------------------------------------
immune system is supported by proper nutrition, sleep, and a healthy lifestyle.
Common Diseases: Common diseases include conditions like diabetes, hypertension, and obesity. 
These are often linked to lifestyle and can be managed through diet, exercise, and medication.
{'creationdate': '2026-03-26T12:09:44+05:30', 'page_label': '1', 'pag

In [ ]:
# Check howmany collection have by db

In [35]:
from chromadb import Client

client = Client()

collections = client.list_collections()

print("Total collections:", len(collections))

for col in collections:
    print(col.name)

Total collections: 1
Mix


In [76]:
from sentence_transformers import SentenceTransformer

# Old model
model_v1 = SentenceTransformer('all-MiniLM-L6-v2')
vec1 = model_v1.encode("AI is powerful")

# New model
model_v2 = SentenceTransformer('BAAI/bge-m3')
vec2 = model_v2.encode("AI is powerful")

print(len(vec1), len(vec2))  # Different dimensions → drift

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3746.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17539.63it/s]


384 1024


In [ ]:
# Metadata Filtering with multiple files

In [3]:
files=[
    ("/home/shrutik/Downloads/aiml.pdf","AI","Research"),
    ("/home/shrutik/Downloads/health.pdf","Wellness","Health")
]
all_docs = []

In [4]:
files

[('/home/shrutik/Downloads/aiml.pdf', 'AI', 'Research'),
 ('/home/shrutik/Downloads/health.pdf', 'Wellness', 'Health')]

In [7]:
for file,dept,topic in files:
    loader = PyPDFLoader(file)
    docs = loader.load()

    for i in docs:
        i.metadata["Dept"] = dept
        i.metadata["File"] = file
        i.metadata["Topic"] = topic

    all_docs.extend(docs)

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=80)

chunked_doc = splitter.split_documents(all_docs)


In [9]:
embeddings = HE(model="BAAI/bge-m3")


/home/shrutik/Srutik/12-Week-Internship/.rag/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
db = Chroma.from_documents(chunked_doc,
                           embeddings,
                           collection_name="Mix")

In [11]:
query = "what is benefit of ai in exercise"

results = db.similarity_search(query,
                               k=3,
                              #  filter={"Topic":"Health"}
                            )

In [12]:
for r in results:
    print("-" * 100)
    print("Content:", r.page_content)
    print("Metadata:", r.metadata)

----------------------------------------------------------------------------------------------------
Content: fatigue and health issues.
Exercise: Exercise is physical activity that improves strength, endurance, and overall health. 
Regular activities like walking, running, and strength training enhance cardiovascular fitness and 
muscle development.
Metadata: {'creationdate': '2026-03-26T12:09:44+05:30', 'Dept': 'Wellness', 'File': '/home/shrutik/Downloads/health.pdf', 'page': 0, 'page_label': '1', 'source': '/home/shrutik/Downloads/health.pdf', 'Topic': 'Health', 'creator': 'Writer', 'producer': 'LibreOffice 7.3', 'total_pages': 1}
----------------------------------------------------------------------------------------------------
Content: metabolism supports body functions and energy balance.
Metadata: {'total_pages': 1, 'creator': 'Writer', 'Dept': 'Wellness', 'Topic': 'Health', 'page': 0, 'producer': 'LibreOffice 7.3', 'File': '/home/shrutik/Downloads/health.pdf', 'creationdate': 

In [ ]:
%pip install sentencepiece

In [4]:
from transformers import pipeline

translator = pipeline(
    "translation_fr_to_en",
    model="Helsinki-NLP/opus-mt-fr-en"
)

print(translator("Merhaba! Oy olarak ne yardımcı olabilirim?")[0]['translation_text'])

Merhaba! Oy olarak n'yardımcı olabilirim?
